# ⚡ LangChain Framework RAG with Groq + Gradio — Colab Notebook

This notebook converts the folder-wise VS Code task into a **Google Colab workflow**.

You will run the complete RAG pipeline:

```text
Data Ingestion → Data Transformation → Embeddings → FAISS Vector DB → Groq Answer → Gradio UI
```

This notebook is suitable for classroom demonstration and student practice.


## Step 1 — Install Required Packages

Run this cell first. It installs LangChain, Groq, FAISS, Hugging Face embeddings, PDF support, and Gradio.


In [3]:
%pip install -qU \
  langchain \
  langchain-core \
  langchain-groq \
  langchain-text-splitters \
  langchain-huggingface \
  langchain-community \
  faiss-cpu \
  sentence-transformers \
  pypdf \
  gradio

## Step 2 — Add Your Groq API Key

Recommended method in Colab:

1. Click **Secrets** from the left sidebar.
2. Add a new secret named:

```text
GROQ_API_KEY
```

3. Paste your Groq API key as the value.
4. Enable notebook access.

If the secret is not found, the notebook will ask you to enter the key securely.


In [4]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    groq_key = userdata.get("GROQ_API_KEY")
except Exception:
    groq_key = None

if not groq_key:
    groq_key = getpass("Enter your Groq API key: ")

os.environ["GROQ_API_KEY"] = groq_key
os.environ["GROQ_MODEL"] = "llama-3.1-8b-instant"
os.environ["EMBEDDING_MODEL"] = "sentence-transformers/all-MiniLM-L6-v2"

print("Groq key loaded:", "Yes" if os.environ.get("GROQ_API_KEY") else "No")
print("Groq model:", os.environ["GROQ_MODEL"])
print("Embedding model:", os.environ["EMBEDDING_MODEL"])

Groq key loaded: Yes
Groq model: llama-3.1-8b-instant
Embedding model: sentence-transformers/all-MiniLM-L6-v2


## Step 3 — Import Libraries

This notebook uses:

- `ChatGroq` for Groq LLM response generation
- `FAISS` for vector search
- `HuggingFaceEmbeddings` for embeddings
- `Gradio` for the web UI
- `pypdf` for PDF reading


In [5]:
from pathlib import Path
import xml.etree.ElementTree as ET

from pypdf import PdfReader

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import gradio as gr

DATA_DIR = Path("/content/data/raw")
VECTOR_DB_PATH = Path("/content/faiss_index")

DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully.")
print("Data folder:", DATA_DIR)

/tmp/ipykernel_901/3898541187.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Libraries imported successfully.
Data folder: /content/data/raw


## Step 4 — Create Sample Data

This creates a sample text document so students can run the RAG pipeline even without uploading a file.


In [6]:
sample_text = """Generative AI Course Notes

Generative AI is a type of artificial intelligence that can create new content such as text, images, code, audio, summaries, and conversations.

LangChain is a framework used to build applications with Large Language Models. It helps connect prompts, models, tools, memory, documents, chains, and retrieval systems.

A RAG system means Retrieval-Augmented Generation. It retrieves relevant information from documents and then generates an answer using an LLM.

The main steps of a simple RAG pipeline are:
1. Data ingestion: load documents.
2. Data transformation: split documents into chunks.
3. Embeddings: convert text chunks into numerical vectors.
4. Vector database: store and search embeddings.
5. Retrieval: find relevant chunks for a user question.
6. LLM answer generation: use retrieved chunks to answer the question.

Groq provides fast inference for open-source language models. Gradio is used to create a simple web interface for AI applications.

Attention is a mechanism used in transformer models. It helps the model focus on the most relevant words or tokens when generating a response.
"""

sample_file = DATA_DIR / "sample_ai_notes.txt"
sample_file.write_text(sample_text, encoding="utf-8")

print("Sample file created:", sample_file)
print("Files in data folder:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Sample file created: /content/data/raw/sample_ai_notes.txt
Files in data folder:
- sample_ai_notes.txt


## Step 5 — Optional: Upload Your Own Files

You can upload PDF, TXT, MD, or XML files.

For classroom practice, students can upload their own notes, reports, or PDFs.


In [7]:
from google.colab import files

uploaded = files.upload()

for filename, content in uploaded.items():
    file_path = DATA_DIR / filename
    file_path.write_bytes(content)
    print("Uploaded:", file_path)

print("\nCurrent files:")
for file in DATA_DIR.iterdir():
    print("-", file.name)

Saving SHCC-ACT-2013.pdf to SHCC-ACT-2013.pdf
Uploaded: /content/data/raw/SHCC-ACT-2013.pdf

Current files:
- SHCC-ACT-2013.pdf
- sample_ai_notes.txt


## Step 6 — Data Ingestion

This step loads documents from the `/content/data/raw` folder.

Supported file types:

- `.txt`
- `.md`
- `.pdf`
- `.xml`


In [8]:
def read_txt_or_md(path: Path) -> str:
    return path.read_text(encoding="utf-8", errors="ignore")


def read_pdf(path: Path):
    reader = PdfReader(str(path))
    docs = []

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text() or ""
        if text.strip():
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "page": page_number,
                        "type": "pdf"
                    }
                )
            )

    return docs


def read_xml(path: Path) -> str:
    try:
        tree = ET.parse(path)
        root = tree.getroot()
        text_parts = []

        for element in root.iter():
            if element.text and element.text.strip():
                text_parts.append(element.text.strip())

        return "\n".join(text_parts)
    except Exception:
        return path.read_text(encoding="utf-8", errors="ignore")


def load_documents_from_directory(directory=DATA_DIR):
    documents = []
    supported_extensions = {".txt", ".md", ".pdf", ".xml"}

    for path in sorted(Path(directory).rglob("*")):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix not in supported_extensions:
            continue

        if suffix == ".pdf":
            documents.extend(read_pdf(path))

        elif suffix in {".txt", ".md"}:
            text = read_txt_or_md(path)
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "type": suffix.replace(".", "")
                    }
                )
            )

        elif suffix == ".xml":
            text = read_xml(path)
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": path.name,
                        "type": "xml"
                    }
                )
            )

    return documents


documents = load_documents_from_directory(DATA_DIR)

print("STEP 1: DATA INGESTION")
print("=" * 50)
print("Documents loaded:", len(documents))

for i, doc in enumerate(documents[:3], start=1):
    print(f"\nDocument {i}")
    print("-" * 50)
    print("Source:", doc.metadata.get("source"))
    print("Type:", doc.metadata.get("type"))
    print(doc.page_content[:500])

STEP 1: DATA INGESTION
Documents loaded: 20

Document 1
--------------------------------------------------
Source: SHCC-ACT-2013.pdf
Type: pdf
PROVINCIAL ASSEMBLY OF SINDH 
NOTIFICATION 
KARACHI, THE 20TH MARCH, 2014. 
 
NO.PAS/Legis-B-09/2013-The Sindh Healthcare Commission Bill, 2013 having been passed 
by the Provincial Assembly of Sindh on 24 th February, 2014 and assented to by the Governor 
of Sindh on 19th March, 2014 is hereby published as an Act of the Legislature of Sindh. 
 
THE SINDH HEALTHCARE COMMISSION ACT, 2013 
 
SINDH ACT NO. VII OF 2014 
 
AN 
ACT 
         to improve the quality of h ealthcare services and banning q

Document 2
--------------------------------------------------
Source: SHCC-ACT-2013.pdf
Type: pdf
 
 
 
2 
 
(x) “Council for Homeopathy” means the National Council for 
Homeopathy established under the Unani, Ayurvedic and 
Homoeopathic Practitioners Act, 1965 (Act II of 1965); 
 
(xi) “Council for Tibb” means the National Council for Tibb 
established

## Step 7 — Data Transformation

Now we split long documents into smaller chunks.

Why chunking is important:

- LLMs cannot always process very long files at once.
- Smaller chunks help retrieval.
- FAISS searches chunks, not the full document.


In [9]:
def split_documents(documents, chunk_size=500, chunk_overlap=120):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    return splitter.split_documents(documents)


chunks = split_documents(documents, chunk_size=500, chunk_overlap=120)

print("STEP 2: DATA TRANSFORMATION")
print("=" * 50)
print("Original documents:", len(documents))
print("Chunks created:", len(chunks))

for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\nChunk {i}")
    print("-" * 50)
    print("Source:", chunk.metadata.get("source"))
    print(chunk.page_content[:500])

STEP 2: DATA TRANSFORMATION
Original documents: 20
Chunks created: 130

Chunk 1
--------------------------------------------------
Source: SHCC-ACT-2013.pdf
PROVINCIAL ASSEMBLY OF SINDH 
NOTIFICATION 
KARACHI, THE 20TH MARCH, 2014. 
 
NO.PAS/Legis-B-09/2013-The Sindh Healthcare Commission Bill, 2013 having been passed 
by the Provincial Assembly of Sindh on 24 th February, 2014 and assented to by the Governor 
of Sindh on 19th March, 2014 is hereby published as an Act of the Legislature of Sindh. 
 
THE SINDH HEALTHCARE COMMISSION ACT, 2013 
 
SINDH ACT NO. VII OF 2014 
 
AN 
ACT

Chunk 2
--------------------------------------------------
Source: SHCC-ACT-2013.pdf
THE SINDH HEALTHCARE COMMISSION ACT, 2013 
 
SINDH ACT NO. VII OF 2014 
 
AN 
ACT 
         to improve the quality of h ealthcare services and banning quackery in the 
Province of Sindh in all its forms and manifestations; 
 
 
         WHEREAS it is expedient to make pr ovision for the improvement, 
access, equity , and  qua

## Step 8 — Embeddings and FAISS Vector Database

This step converts text chunks into numerical vectors.

Then FAISS stores those vectors for fast similarity search.

The first run may take some time because the embedding model will download.


In [10]:
EMBEDDING_MODEL = os.environ.get(
    "EMBEDDING_MODEL",
    "sentence-transformers/all-MiniLM-L6-v2"
)

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vector_store = FAISS.from_documents(chunks, embeddings)
vector_store.save_local(str(VECTOR_DB_PATH))

print("STEP 3: EMBEDDINGS + FAISS")
print("=" * 50)
print("Embedding model:", EMBEDDING_MODEL)
print("Chunks embedded:", len(chunks))
print("Vector database saved at:", VECTOR_DB_PATH)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

STEP 3: EMBEDDINGS + FAISS
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Chunks embedded: 130
Vector database saved at: /content/faiss_index


## Step 9 — Test Vector Search

Before using Groq, test whether FAISS can retrieve relevant chunks.


In [12]:
question = "what is the rule of getting casual leave?"

results = vector_store.similarity_search(question, k=3)

print("STEP 4: VECTOR SEARCH")
print("=" * 50)
print("Question:", question)

for i, doc in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("-" * 50)
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:700])

STEP 4: VECTOR SEARCH
Question: what is the rule of getting casual leave?

Result 1
--------------------------------------------------
Source: SHCC-ACT-2013.pdf
(2) In case of a casual vacancy of a Commissioner, Government shall 
appoint a person as Commissioner in acc ordance with the provisions of section 
5 for the remainder of the term of the Commissioner, who has died, resigned or 
disqualified under this Act.  
 
 
Term of the 
Commissioners.  
 
 
7.       No person shall be, or sh all continue to be, the Chairman or a 
Commissioner who - 
(a) has tendered resignation and no t withdrawn it within a period 
of thirty days;

Result 2
--------------------------------------------------
Source: SHCC-ACT-2013.pdf
Commissioner who - 
(a) has tendered resignation and no t withdrawn it within a period 
of thirty days; 
(b) is, or at any time has been, adjudicated as insolvent; 
 
(c) is found to be of unsound mind by a court of competent 
jurisdiction; 
 
(d) is, or has at any time been,

## Step 10 — Create Groq RAG Chain

Now we combine:

```text
User Question → FAISS Retrieval → Context → Groq LLM → Final Answer
```


In [14]:
def get_llm(temperature=0.1):
    return ChatGroq(
        model=os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant"),
        temperature=temperature,
        api_key=os.environ.get("GROQ_API_KEY")
    )


def answer_with_groq(question, k=4, temperature=0.1):
    retrieved_docs = vector_store.similarity_search(question, k=k)

    context = "\n\n".join(
        f"[Source: {doc.metadata.get('source', 'unknown')}] {doc.page_content}"
        for doc in retrieved_docs
    )

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful Legal Advisor. "
                "Answer using only the given context. "
                "Use legal terms in the language. "
                "If the answer is not in the context, say: "
                "'I do not know from the uploaded documents.'"
            ),
            (
                "human",
                "Context:\n{context}\n\nQuestion:\n{question}\n\n"
                "Give a clear answer."
            )
        ]
    )

    llm = get_llm(temperature=temperature)
    chain = prompt | llm | StrOutputParser()

    answer = chain.invoke(
        {
            "context": context,
            "question": question
        }
    )

    sources = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "unknown")
        page = doc.metadata.get("page")
        label = f"{source}, page {page}" if page else source
        if label not in sources:
            sources.append(label)

    return answer, sources


answer, sources = answer_with_groq("What is Quackery?", k=3)

print("Groq Answer")
print("=" * 50)
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Groq Answer
According to Section (xxix) of the Sindh Healthcare Commission Act, 2013, "quack" means a pretender providing health services without having registration of Pakistan Medical Dental Council, Council for Tibb and Homeopathy and Nursing Council.

Sources:
- SHCC-ACT-2013.pdf, page 1
- SHCC-ACT-2013.pdf, page 3
- SHCC-ACT-2013.pdf, page 5


## Step 11 — Ask Your Own Question

Change the question below and run the cell again.


In [15]:
my_question = "What is SHCC mandate?"

answer, sources = answer_with_groq(my_question, k=4, temperature=0.1)

print("Question:", my_question)
print("\nAnswer:")
print(answer)

print("\nSources:")
for source in sources:
    print("-", source)

Question: What is SHCC mandate?

Answer:
The SHCC mandate, as per the Sindh Healthcare Commission Act, 2013, includes the following key responsibilities:

1. Maintaining a register of all healthcare service providers.
2. Granting, revoking, and renewing licenses to persons involved in the provision of healthcare services.
3. Regulating the quality and standards of healthcare services.
4. Regulating the prices of healthcare services.
5. Establishing and regulating quality assurance committees in healthcare establishments.

These mandates are aimed at ensuring the provision of quality healthcare services in the Province of Sindh.

Sources:
- SHCC-ACT-2013.pdf, page 4
- SHCC-ACT-2013.pdf, page 1
- SHCC-ACT-2013.pdf, page 3
- SHCC-ACT-2013.pdf, page 18


## Step 12 — Groq Chat Playground

This is a simple Groq chat without RAG.

Use it to explain prompting and temperature.


In [18]:
def simple_groq_chat(message, temperature=0.4):
    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "You are a helpful AI Legal Advisor of a health regulator. "
                "Explain concepts simply for health professionals."
            ),
            ("human", "{message}")
        ]
    )

    llm = get_llm(temperature=temperature)
    chain = prompt | llm | StrOutputParser()

    return chain.invoke({"message": message})


print(simple_groq_chat("Explain how to register a HCE?", temperature=0.4))

As a health regulator, I'd be happy to explain how to register a High-Complexity Establishment (HCE) in a simple and straightforward manner.

**What is a High-Complexity Establishment (HCE)?**

A High-Complexity Establishment (HCE) is a medical facility or organization that performs high-risk medical procedures or handles high-risk medical products. Examples of HCEs include hospitals, surgical centers, and laboratories that perform complex tests or procedures.

**Registration Requirements:**

To register an HCE, you'll need to meet the following requirements:

1. **Licensing:** Ensure that your HCE has the necessary licenses and permits to operate in your jurisdiction.
2. **Accreditation:** Obtain accreditation from a recognized accrediting agency, such as The Joint Commission or the College of American Pathologists (CAP).
3. **Compliance with Regulations:** Ensure that your HCE complies with all relevant laws, regulations, and standards, including those related to patient safety, qual

## Step 13 — Appealing Gradio UI App

This app has two tabs:

1. **Ask Your Documents** — RAG-based question answering
2. **Groq Chat Playground** — normal Groq chatbot

In Colab, `share=True` creates a public temporary Gradio link.


In [19]:
CUSTOM_CSS = """
.gradio-container {
    max-width: 1150px !important;
    margin: auto !important;
}
#hero {
    padding: 26px;
    border-radius: 22px;
    background: linear-gradient(135deg, #0f172a, #1e293b, #172554);
    color: white;
    box-shadow: 0 18px 60px rgba(0,0,0,.22);
    margin-bottom: 18px;
}
#hero h1 {
    font-size: 38px;
    margin-bottom: 8px;
    background: linear-gradient(90deg, #22c55e, #38bdf8, #a78bfa);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
#hero p {
    color: #cbd5e1;
    font-size: 16px;
}
"""


def rag_ui_answer(question, k, temperature):
    if not question or not question.strip():
        return "Please write a question."

    try:
        answer, sources = answer_with_groq(
            question=question,
            k=int(k),
            temperature=float(temperature)
        )

        source_text = "\n".join(f"- {source}" for source in sources)

        return f"""{answer}

### Sources
{source_text}
"""

    except Exception as e:
        return f"""Error: {str(e)}

Please check:
1. Groq API key is set
2. Documents are loaded
3. Vector database was created
"""


def playground_ui(message, temperature):
    if not message or not message.strip():
        return "Please write a prompt."

    try:
        return simple_groq_chat(message, temperature=float(temperature))
    except Exception as e:
        return f"Error: {str(e)}"


with gr.Blocks(css=CUSTOM_CSS, title="Groq RAG Studio", theme=gr.themes.Soft()) as demo:
    gr.HTML(
        """
        <div id="hero">
            <h1>⚡ Groq RAG Studio</h1>
            <p>Google Colab version of LangChain Framework using Groq API, FAISS, embeddings, and Gradio.</p>
            <p>Ask questions from your uploaded notes, PDFs, TXT, MD, or XML files.</p>
        </div>
        """
    )

    with gr.Tabs():
        with gr.Tab("📚 Ask Your Documents"):
            gr.Markdown("Ask questions from the documents loaded into FAISS.")

            question_box = gr.Textbox(
                label="Your Question",
                placeholder="Example: What is RAG? What is LangChain? What is attention?",
                lines=3
            )

            with gr.Row():
                k_slider = gr.Slider(1, 8, value=4, step=1, label="Retrieved Chunks")
                temp_slider = gr.Slider(0, 1, value=0.1, step=0.1, label="Temperature")

            ask_btn = gr.Button("Ask Groq", variant="primary")
            output_box = gr.Markdown(label="Answer")

            ask_btn.click(
                fn=rag_ui_answer,
                inputs=[question_box, k_slider, temp_slider],
                outputs=output_box
            )

        with gr.Tab("🧠 Groq Chat Playground"):
            gr.Markdown("Use this tab to test normal Groq prompting without documents.")

            prompt_box = gr.Textbox(
                label="Prompt",
                placeholder="Explain prompt engineering in simple words.",
                lines=5
            )

            playground_temp = gr.Slider(0, 1, value=0.4, step=0.1, label="Temperature")
            generate_btn = gr.Button("Generate with Groq", variant="primary")
            playground_output = gr.Textbox(label="Groq Response", lines=12)

            generate_btn.click(
                fn=playground_ui,
                inputs=[prompt_box, playground_temp],
                outputs=playground_output
            )

        with gr.Tab("🧩 RAG Pipeline Explanation"):
            gr.Markdown(
                """
                ## RAG Pipeline Used in This Notebook

                1. **Data Ingestion**
                   Load PDF, TXT, MD, and XML files.

                2. **Data Transformation**
                   Split long text into smaller chunks.

                3. **Embeddings**
                   Convert chunks into numerical vectors.

                4. **FAISS Vector Database**
                   Store and search similar chunks.

                5. **Retrieval**
                   Retrieve the most relevant chunks for the user's question.

                6. **Groq LLM**
                   Generate an answer using retrieved context.

                7. **Gradio UI**
                   Provide an easy web interface.
                """
            )

demo.launch(share=True)

/tmp/ipykernel_901/1210065548.py:67: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Groq RAG Studio", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f8cb830075bd40a82f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Student Task

Students should try the following:

1. Upload a PDF or TXT file.
2. Re-run the ingestion cell.
3. Re-run the transformation cell.
4. Re-run the embedding cell.
5. Ask questions from the uploaded document.
6. Compare answers at different temperature values.
7. Explain each RAG step in their own words.
